# RoPE Adaptation & Fine-Tuning Suite (16 Models)

This notebook fine-tunes pretrained checkpoints to adapt them to **Rotary Positional Embeddings (RoPE)**, **Weight Tying**, and **Gradient Checkpointing**.

### Models Covered:
- **12.5M**: 1:1, 1:5, 1:10, 1:15, 1:20, 1:25 (6 models)
- **25M**: 1:1, 1:10, 1:15, 1:20, 1:25, 1:30 (6 models)
- **50M**: 1:25, 1:35, 1:40, 1:45 (4 models)

**Fine-Tuning Budget**: 150 steps per model

In [ ]:
import os
import sys
import time
import gc
import yaml
from pathlib import Path

# Ensure project root is the working directory
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
from telos.model.mlx_components import MLXTelosTransformer
from telos.training.trainer import TelosMLXTrainer

print(f"Project root resolved: {project_root}")
print(f"MLX Device: {mx.default_device()}")

In [ ]:
def finetune_rope_model(
    config_path: str,
    source_ckpt: str,
    output_dir_name: str,
    steps: int = 150,
    max_lr: float = 1e-4,
    min_lr: float = 1e-5,
    warmup_steps: int = 15
):
    """Fine-tunes a pretrained model to adapt attention heads to RoPE frequencies."""
    print("=" * 85)
    print(f"[RoPE Fine-Tune] Starting: {output_dir_name}")
    print(f"  Config: {config_path}")
    print(f"  Source Weights: {source_ckpt}")
    print(f"  Target Steps: {steps} | Max LR: {max_lr:.1e} | Min LR: {min_lr:.1e}")
    print("=" * 85)
    
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
        
    # Configure fine-tuning parameters in memory
    cfg["training"]["max_steps"] = steps
    cfg["training"]["warmup_steps"] = warmup_steps
    cfg["training"]["max_lr"] = max_lr
    cfg["training"]["min_lr"] = min_lr
    cfg["checkpoint"]["checkpoint_dir"] = f"checkpoints/{output_dir_name}"
    cfg["checkpoint"]["save_every_steps"] = steps
    
    # Initialize model with RoPE and tied embeddings
    model = MLXTelosTransformer(**cfg["model"])
    model.set_dtype(mx.bfloat16)
    
    # Load pretrained weights (strict=False ignores old head weights for tied embeddings)
    print(f"  Loading source checkpoint: {source_ckpt} ...")
    model.load_weights(source_ckpt, strict=False)
    
    # Launch trainer
    trainer = TelosMLXTrainer(model, cfg)
    t_start = time.time()
    trainer.train(resume_step=0)
    elapsed = (time.time() - t_start) / 60.0
    
    # Clean up Metal unified memory
    del model, trainer
    gc.collect()
    mx.clear_cache()
    
    print(f"[RoPE Fine-Tune] Completed {output_dir_name} in {elapsed:.2f} minutes.\n")
    return elapsed

In [ ]:
# Registry of all models with their source checkpoints
MODELS_TO_FINETUNE = [
    # 12.5M Models
    {
        "name": "12.5M 1:1",
        "config": "configs/phase_b_12m_1to1_mlx.yaml",
        "source": "checkpoints/phase_b_12m_1to1_mlx_20260808_230424/model.safetensors",
        "output": "phase_b_12m_1to1_mlx_rope_ft"
    },
    {
        "name": "12.5M 1:5",
        "config": "configs/phase_b_12m_1to5_mlx.yaml",
        "source": "checkpoints/phase_b_12m_1to5_mlx_20260808_230632/model.safetensors",
        "output": "phase_b_12m_1to5_mlx_rope_ft"
    },
    {
        "name": "12.5M 1:10",
        "config": "configs/phase_b_12m_1to10_mlx.yaml",
        "source": "checkpoints/phase_b_12m_1to10_mlx_20260808_231924/model.safetensors",
        "output": "phase_b_12m_1to10_mlx_rope_ft"
    },
    {
        "name": "12.5M 1:15",
        "config": "configs/phase_b_12m_1to15_mlx.yaml",
        "source": "checkpoints/phase_b_12m_1to15_mlx_20260808_234930/model.safetensors",
        "output": "phase_b_12m_1to15_mlx_rope_ft"
    },
    {
        "name": "12.5M 1:20",
        "config": "configs/phase_b_12m_1to20_mlx.yaml",
        "source": "checkpoints/phase_b_12m_1to20_mlx_20260809_002244/model.safetensors",
        "output": "phase_b_12m_1to20_mlx_rope_ft"
    },
    {
        "name": "12.5M 1:25",
        "config": "configs/phase_b_12m_1to25_mlx.yaml",
        "source": "checkpoints/phase_b_12m_1to25_mlx_20260809_010631/model.safetensors",
        "output": "phase_b_12m_1to25_mlx_rope_ft"
    },
    
    # 25M Models
    {
        "name": "25M 1:1",
        "config": "configs/phase_b_25m_1to1_mlx.yaml",
        "source": "checkpoints/phase_b_25m_1to1_mlx_20260809_074837/model.safetensors",
        "output": "phase_b_25m_1to1_mlx_rope_ft"
    },
    {
        "name": "25M 1:10",
        "config": "configs/phase_b_25m_1to10_mlx.yaml",
        "source": "checkpoints/phase_b_25m_1to10_mlx_20260809_075542/model.safetensors",
        "output": "phase_b_25m_1to10_mlx_rope_ft"
    },
    {
        "name": "25M 1:15",
        "config": "configs/phase_b_25m_1to15_mlx.yaml",
        "source": "checkpoints/phase_b_25m_1to15_mlx_20260809_090349/model.safetensors",
        "output": "phase_b_25m_1to15_mlx_rope_ft"
    },
    {
        "name": "25M 1:20",
        "config": "configs/phase_b_25m_1to20_mlx.yaml",
        "source": "checkpoints/phase_b_25m_1to20_mlx_20260809_104139/model.safetensors",
        "output": "phase_b_25m_1to20_mlx_rope_ft"
    },
    {
        "name": "25M 1:25",
        "config": "configs/phase_b_25m_1to25_mlx.yaml",
        "source": "checkpoints/phase_b_25m_1to25_mlx_20260809_131404/model.safetensors",
        "output": "phase_b_25m_1to25_mlx_rope_ft"
    },
    {
        "name": "25M 1:30",
        "config": "configs/phase_b_25m_1to30_mlx.yaml",
        "source": "checkpoints/phase_b_25m_1to30_mlx/model.safetensors",
        "output": "phase_b_25m_1to30_mlx_rope_ft"
    },
    
    # 50M Models
    {
        "name": "50M 1:25",
        "config": "configs/phase_b_50m_1to25_mlx.yaml",
        "source": "checkpoints/phase_b_50m_1to25_mlx_20260809_225226/model.safetensors",
        "output": "phase_b_50m_1to25_mlx_rope_ft"
    },
    {
        "name": "50M 1:35",
        "config": "configs/phase_b_50m_1to35_mlx.yaml",
        "source": "checkpoints/phase_b_50m_1to35_mlx_20260811_231951/model.safetensors",
        "output": "phase_b_50m_1to35_mlx_rope_ft"
    },
    {
        "name": "50M 1:40",
        "config": "configs/phase_b_50m_1to40_mlx.yaml",
        "source": "checkpoints/phase_b_50m_1to40_mlx_20260813_222750/model.safetensors",
        "output": "phase_b_50m_1to40_mlx_rope_ft"
    },
    {
        "name": "50M 1:45",
        "config": "configs/phase_b_50m_1to45_mlx.yaml",
        "source": "checkpoints/phase_b_50m_1to45_mlx_20260816_001417/model.safetensors",
        "output": "phase_b_50m_1to45_mlx_rope_ft"
    }
]

print(f"Registered {len(MODELS_TO_FINETUNE)} models for RoPE fine-tuning.")

In [ ]:
# TARGETED RUN: 50M 1:45 ROPE FINE-TUNING (150 Steps, ~15 mins)
print("\n>>> Fine-tuning 50M 1:45 for RoPE Adaptation <<<")
finetune_rope_model(
    config_path="configs/phase_b_50m_1to45_mlx.yaml",
    source_ckpt="checkpoints/phase_b_50m_1to45_mlx_20260816_001417/model.safetensors",
    output_dir_name="phase_b_50m_1to45_mlx_rope_ft",
    steps=150,
    max_lr=1e-4,
    min_lr=1e-5,
    warmup_steps=15
)

In [ ]:
# OPTIONAL: EXECUTE COMPLETE 16-MODEL BATCH RUN
# total_suite_start = time.time()
# timing_log = {}
# for idx, item in enumerate(MODELS_TO_FINETUNE, 1):
#     print(f"\n>>> [{idx}/{len(MODELS_TO_FINETUNE)}] Fine-tuning {item['name']} <<<")
#     elapsed_min = finetune_rope_model(
#         config_path=item["config"],
#         source_ckpt=item["source"],
#         output_dir_name=item["output"],
#         steps=150,
#         max_lr=1e-4,
#         min_lr=1e-5,
#         warmup_steps=15
#     )
#     timing_log[item["name"]] = elapsed_min
# print(f"Completed in {(time.time() - total_suite_start)/3600.0:.2f} hours.")